<a href="https://colab.research.google.com/github/Loubna1101/Qwen3-TTS-Darija-LoRa/blob/main/notebooks/finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Step 1: Install dependencies**


In [ ]:
!pip install -q qwen-tts peft bitsandbytes soundfile librosa
!git clone https://github.com/QwenLM/Qwen3-TTS.git

**Step 2: Mount Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Step 3: Project paths**

In [ ]:
from pathlib import Path
import shutil

PROJECT_NAME = "qwen3_darija_tts"

RAW_DATA_DIR = Path("/content/drive/MyDrive/Qwen-Dataset")
PROC_DATA_DIR = Path("/content/drive/MyDrive/Qwen-Dataset_24k_2")
OUTPUT_DIR = Path("/content/drive/MyDrive/qwen3_darija_tts_output_2")

METADATA_FILE0 = RAW_DATA_DIR / "metadata.csv"
METADATA_FILE = PROC_DATA_DIR / "metadata.csv"

shutil.copy2(METADATA_FILE0, METADATA_FILE)
REF_AUDIO = PROC_DATA_DIR / "ref.wav"

TRAIN_RAW_JSONL = PROC_DATA_DIR / "train_raw_24k.jsonl"
TRAIN_CODES_JSONL = PROC_DATA_DIR / "train_with_codes.jsonl"
TRAIN_CODES_CLEAN_JSONL = PROC_DATA_DIR / "train_with_codes_clean.jsonl"

PROC_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROC_DATA_DIR:", PROC_DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

**Step 4: Resample audio to 24KHz**

In [ ]:
import librosa
import soundfile as sf

count = 0

for wav_path in RAW_DATA_DIR.glob("*.wav"):
    audio, sr = librosa.load(wav_path, sr=24000, mono=True)
    out_path = PROC_DATA_DIR / wav_path.name
    sf.write(out_path, audio, 24000)
    count += 1

print(f"Done. Resampled {count} files to {PROC_DATA_DIR}")

In [ ]:
audio_path = PROC_DATA_DIR / "clip_5000.wav"  # change if needed
data, sr = sf.read(audio_path)
print("Sample rate:", sr)
print("Shape:", data.shape)

**Step 5: Build JSONL**

In [ ]:
import csv
import json

count = 0

with open(METADATA_FILE, "r", encoding="utf-8-sig") as f_in, open(TRAIN_RAW_JSONL, "w", encoding="utf-8") as f_out:
    reader = csv.DictReader(f_in)
    for row in reader:
        audio_name = row["audio_name"].strip()
        text = row["transcription"].strip()
        audio_path = PROC_DATA_DIR / audio_name

        if not audio_path.exists():
            print("Missing:", audio_path)
            continue

        if audio_path.resolve() == REF_AUDIO.resolve():
            continue

        item = {
            "audio": str(audio_path),
            "text": text,
            "ref_audio": str(REF_AUDIO)
        }
        f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
        count += 1

print(f"Done. Wrote {count} lines to {TRAIN_RAW_JSONL}")

**Step 6: Prepare codec tokens**

In [ ]:
!apt-get update && apt-get install -y sox libsox-fmt-all

In [ ]:
%cd /content/Qwen3-TTS/finetuning

In [ ]:
!python prepare_data.py \
  --device cuda:0 \
  --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz \
  --input_jsonl {TRAIN_RAW_JSONL} \
  --output_jsonl {TRAIN_CODES_JSONL}

**Step 7: Clean malformed entries**

In [ ]:
import json

total = 0
kept = 0
removed = 0

with open(TRAIN_CODES_JSONL, "r", encoding="utf-8") as fin, open(TRAIN_CODES_CLEAN_JSONL, "w", encoding="utf-8") as fout:
    for line in fin:
        total += 1
        obj = json.loads(line)
        codes = obj.get("audio_codes", None)

        if not codes:
            removed += 1
            continue

        if not isinstance(codes, list) or len(codes) == 0 or not isinstance(codes[0], list):
            removed += 1
            continue

        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
        kept += 1

print("Total:", total)
print("Kept:", kept)
print("Removed:", removed)
print("Clean file:", TRAIN_CODES_CLEAN_JSONL)

**Step 8: Modify the training script**

In [ ]:
%%writefile /content/Qwen3-TTS/finetuning/sft_12hz.py
# coding=utf-8
# Copyright 2026 The Alibaba Qwen team.
# SPDX-License-Identifier: Apache-2.0
#
# Modified for LoRA-based custom voice fine-tuning and Colab-friendly saving.
#
# Licensed under the Apache License, Version 2.0.

import argparse
import json
import os

import torch
from accelerate import Accelerator
from dataset import TTSDataset
from peft import LoraConfig, get_peft_model
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoConfig

target_speaker_embedding = None


def train():
    global target_speaker_embedding

    parser = argparse.ArgumentParser()
    parser.add_argument("--init_model_path", type=str, default="Qwen/Qwen3-TTS-12Hz-1.7B-Base")
    parser.add_argument("--output_model_path", type=str, default="output")
    parser.add_argument("--train_jsonl", type=str, required=True)
    parser.add_argument("--batch_size", type=int, default=1)
    parser.add_argument("--lr", type=float, default=5e-5)
    parser.add_argument("--num_epochs", type=int, default=3)
    parser.add_argument("--speaker_name", type=str, default="speaker_test")

    parser.add_argument("--lora_r", type=int, default=8)
    parser.add_argument("--lora_alpha", type=int, default=16)
    parser.add_argument("--lora_dropout", type=float, default=0.05)

    args = parser.parse_args()

    accelerator = Accelerator(
        gradient_accumulation_steps=4,
        mixed_precision="fp16"
    )

    model_path = args.init_model_path

    qwen3tts = Qwen3TTSModel.from_pretrained(
        model_path,
        attn_implementation="eager",
    )
    config = AutoConfig.from_pretrained(model_path)

    lora_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=args.lora_dropout,
        bias="none",
    )

    qwen3tts.model.talker.model = get_peft_model(qwen3tts.model.talker.model, lora_config)
    qwen3tts.model.talker.model.print_trainable_parameters()

    with open(args.train_jsonl, "r", encoding="utf-8") as f:
        train_data = [json.loads(line) for line in f]

    dataset = TTSDataset(train_data, qwen3tts.processor, config)
    train_dataloader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=True,
        collate_fn=dataset.collate_fn
    )

    optimizer = AdamW(
        filter(lambda p: p.requires_grad, qwen3tts.model.parameters()),
        lr=args.lr,
        weight_decay=0.01,
    )

    model, optimizer, train_dataloader = accelerator.prepare(
        qwen3tts.model, optimizer, train_dataloader
    )

    model.train()

    for epoch in range(args.num_epochs):
        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(model):
                input_ids = batch["input_ids"]
                codec_ids = batch["codec_ids"]
                ref_mels = batch["ref_mels"]
                text_embedding_mask = batch["text_embedding_mask"]
                codec_embedding_mask = batch["codec_embedding_mask"]
                attention_mask = batch["attention_mask"]
                codec_0_labels = batch["codec_0_labels"]
                codec_mask = batch["codec_mask"]

                speaker_embedding = model.speaker_encoder(
                    ref_mels.to(model.device).to(model.dtype)
                ).detach()

                if target_speaker_embedding is None:
                    target_speaker_embedding = speaker_embedding

                input_text_ids = input_ids[:, :, 0]
                input_codec_ids = input_ids[:, :, 1]

                input_text_embedding = model.talker.model.text_embedding(input_text_ids) * text_embedding_mask
                input_codec_embedding = model.talker.model.codec_embedding(input_codec_ids) * codec_embedding_mask
                input_codec_embedding[:, 6, :] = speaker_embedding

                input_embeddings = input_text_embedding + input_codec_embedding

                for i in range(1, 16):
                    codec_i_embedding = model.talker.code_predictor.get_input_embeddings()[i - 1](
                        codec_ids[:, :, i]
                    )
                    codec_i_embedding = codec_i_embedding * codec_mask.unsqueeze(-1)
                    input_embeddings = input_embeddings + codec_i_embedding

                outputs = model.talker(
                    inputs_embeds=input_embeddings[:, :-1, :],
                    attention_mask=attention_mask[:, :-1],
                    labels=codec_0_labels[:, 1:],
                    output_hidden_states=True
                )

                hidden_states = outputs.hidden_states[0][-1]
                talker_hidden_states = hidden_states[codec_mask[:, :-1]]
                talker_codec_ids = codec_ids[codec_mask]

                _, sub_talker_loss = model.talker.forward_sub_talker_finetune(
                    talker_codec_ids, talker_hidden_states
                )

                loss = outputs.loss + 0.3 * sub_talker_loss

                accelerator.backward(loss)

                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(), 1.0)

                optimizer.step()
                optimizer.zero_grad()

            if step % 10 == 0:
                accelerator.print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")

        if accelerator.is_main_process:
            output_dir = os.path.join(args.output_model_path, f"checkpoint-epoch-{epoch}")
            os.makedirs(output_dir, exist_ok=True)

            qwen3tts.processor.save_pretrained(output_dir)

            output_config_file = os.path.join(output_dir, "config.json")
            config_dict = config.to_dict()
            config_dict["tts_model_type"] = "custom_voice"

            talker_config = config_dict.get("talker_config", {})
            talker_config["spk_id"] = {args.speaker_name: 3000}
            talker_config["spk_is_dialect"] = {args.speaker_name: False}
            config_dict["talker_config"] = talker_config

            with open(output_config_file, "w", encoding="utf-8") as f:
                json.dump(config_dict, f, indent=2, ensure_ascii=False)

            unwrapped_model = accelerator.unwrap_model(model)
            adapter_dir = os.path.join(output_dir, "talker_lora")
            unwrapped_model.talker.model.save_pretrained(adapter_dir)

            if target_speaker_embedding is not None:
                torch.save(
                    {
                        "speaker_name": args.speaker_name,
                        "speaker_id": 3000,
                        "embedding": target_speaker_embedding[0].detach().cpu(),
                    },
                    os.path.join(output_dir, "speaker_embedding.pt")
                )

            metadata = {
                "base_model": args.init_model_path,
                "speaker_name": args.speaker_name,
                "speaker_id": 3000,
                "epoch": epoch,
                "lora": {
                    "r": args.lora_r,
                    "alpha": args.lora_alpha,
                    "dropout": args.lora_dropout,
                    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
                }
            }

            with open(os.path.join(output_dir, "training_meta.json"), "w", encoding="utf-8") as f:
                json.dump(metadata, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    train()

**Step 9: Fine-tune**

In [ ]:
%cd /content/Qwen3-TTS/finetuning

In [ ]:
!python sft_12hz.py \
  --init_model_path Qwen/Qwen3-TTS-12Hz-1.7B-Base \
  --output_model_path /content/drive/MyDrive/qwen3_darija_tts_output \
  --train_jsonl /content/drive/MyDrive/Qwen-Dataset_24k/train_with_codes.jsonl \
  --batch_size 1 \
  --lr 5e-5 \
  --num_epochs 10 \
  --speaker_name darija_speaker